# Interactive Formation Channel Figures (Fig 3, 4, 5)

Interactive Plotly versions of Figures 3, 4, and 5 from the formation channel paper.
Hover over any bar or point to see the Paper name and all formation channel fractions.

**Output:** HTML files saved to `Rates_of_Formation_Channels/interactive_figures_and_tables/`

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
# ── Paths ───────────────────────────────────────────────────────────────────
path_github  = "/Users/floorbroekgaarden/Projects/GitHub/Rates_of_Formation_Channels/"
path_fc_data = path_github + "fc_data/Data_formation_channels_intrinsic/"
path_table   = path_github + "interactive_figures_and_tables/formation_channel_rates_table.csv"
path_save    = path_github + "interactive_figures_and_tables/"

In [ ]:
# ── Colour palette (matches default_scripts.py) ──────────────────────────────
PALETTE = {
    "gray_light":  "#e0e0e0",
    "gray_dark":   "#C2C2C2",
    "orange_main": "#FFA630",
    "orange_sub":  "#ffc857",
    "orange_soft": "#f4a259",
    "red_soft":    "#e45756",
    "rose":        "#d96c75",
    "terracotta":  "#bc4b51",
    "peach":       "#ffb5a7",
    "violet":      "#7b2cbf",
    "purple":      "#9d4edd",
    "magenta":     "#c77dff",
    "lavender":    "#b8a0ff",
    "burgundy":    "#7f1d1d",
    "wine":        "#6d213c",
    "maroon":      "#800020",
    "blue_main":   "#00A7E1",
    "blue_light":  "#5dade2",
    "blue_mid":    "#0474BA",
    "blue_dark":   "#004e89",
    "indigo":      "#3a3d98",
    "teal_light":  "#20b2aa",
    "green_blue":  "#58AD8B",
    "aqua":        "#00bfc4",
    "seafoam":     "#8dd3c7",
    "mint":        "#9de0ad",
}

# ── Channel definitions ──────────────────────────────────────────────────────
CE_SIMPLE_CHANNELS = [
    {"name": "fraction without common envelope", "color": PALETTE["orange_main"], "label": "without CE"},
    {"name": "fraction not specified",           "color": PALETTE["gray_dark"],   "label": "other (CE not specified)"},
    {"name": "fraction with common envelope",    "color": PALETTE["blue_main"],   "label": "with CE"},
]

CE_DETAILED_CHANNELS = [
    {"name": "without CE (no detail specified)",               "color": PALETTE["orange_main"], "label": "without CE (broad)"},
    {"name": "SMT before and after channel",                   "color": PALETTE["orange_sub"],  "label": "classic SMT (SMT+SMT)"},
    {"name": "SMT+NON",                                        "color": PALETTE["rose"],        "label": "SMT + NON"},
    {"name": "NON+SMT",                                        "color": PALETTE["red_soft"],    "label": "NON + SMT"},
    {"name": "channel V intrinsic (z=0) other without CE",    "color": PALETTE["peach"],       "label": "other without CE"},
    {"name": "NON + NON",                                      "color": PALETTE["magenta"],     "label": "NON + NON"},
    {"name": "CHE",                                            "color": PALETTE["lavender"],    "label": "CHE (no MT)"},
    {"name": "channel V intrinsic (z=0) CE not specified",    "color": PALETTE["gray_dark"],   "label": "other (CE not specified)"},
    {"name": "with CE (no detail specified)",                  "color": PALETTE["blue_light"],  "label": "with CE (broad)"},
    {"name": "channel I intrinsic (z=0) classic CE (SMT+CE)", "color": PALETTE["blue_main"],   "label": "classic CE (SMT+CE)"},
    {"name": "channel III intrinsic (z=0) SCCE",              "color": PALETTE["blue_mid"],    "label": "single-core CE (SCCE)"},
    {"name": "channel IV intrinsic (z=0) DCCE",               "color": PALETTE["teal_light"],  "label": "double-core CE (DCCE)"},
    {"name": "CEE + SMT",                                      "color": PALETTE["green_blue"],  "label": "CE + SMT"},
    {"name": "CEE+CEE",                                        "color": PALETTE["aqua"],        "label": "CE + CE"},
    {"name": "NON+CEE",                                        "color": PALETTE["indigo"],      "label": "NON + CE"},
    {"name": "CEE+NON",                                        "color": PALETTE["mint"],        "label": "CE + NON"},
    {"name": "channel radCEE",                                 "color": PALETTE["blue_dark"],   "label": "radiative CE"},
    {"name": "channel convCEE",                                "color": PALETTE["seafoam"],     "label": "convective CE"},
    {"name": "channel V intrinsic (z=0) other with CE",       "color": PALETTE["seafoam"],     "label": "other with CE"},
]

TOTAL_RATE_COL = "All intrinsic (z=0) [Gpc^-3 yr^-1]"

# Observed GW merger rate bands (GWTC-5 population; Table 2)
GW_RATE_BANDS = {
    "BH-BH": (27.5,  49.4),
    "BH-NS": (6.7,   32.8),
    "NS-NS": (5.1,  154.7),
}

# x-axis limits for rate panel
XLIMS_RATE = {
    "BH-BH": (0.5,   600),
    "BH-NS": (0.2,  5000),
    "NS-NS": (0.5,  2000),
}

# DCO label → table column prefix
DCO_PREFIX = {"BH-BH": "BHBH", "BH-NS": "BHNS", "NS-NS": "NSNS"}

# Hover columns per DCO type (from formation_channel_rates_table.csv)
HOVER_COLS = [
    ("{p}_rate_Gpc3yr",       "Rate [Gpc⁻³ yr⁻¹]"),
    ("{p}_frac_CHE",          "frac CHE"),
    ("{p}_frac_SMT",          "frac SMT"),
    ("{p}_frac_other_noCE",   "frac other (no CE)"),
    ("{p}_frac_classicCE",    "frac classic CE"),
    ("{p}_frac_SCCE",         "frac SCCE"),
    ("{p}_frac_DCCE",         "frac DCCE"),
    ("{p}_frac_other_CE",     "frac other CE"),
    ("{p}_total_frac_noCE",   "total frac without CE"),
    ("{p}_total_frac_CE",     "total frac with CE"),
]

In [ ]:
# ── Data helpers ─────────────────────────────────────────────────────────────

def load_all_data(dco_label):
    """Load rates CSV, specs CSV, and the summary table CSV for one DCO type."""
    rates_df = pd.read_csv(path_fc_data + dco_label + "_rates_review.csv").fillna(0.0)
    specs_df = pd.read_csv(path_fc_data + "simulation_specs.csv")
    table_df = pd.read_csv(path_table)
    return rates_df, specs_df, table_df


def filter_to_detailed_subset(df):
    """Keep only rows that have at least one detailed (non-broad) CE/non-CE channel."""
    broad_cols = {
        "fraction without common envelope",
        "fraction with common envelope",
        "fraction not specified",
        "without CE (no detail specified)",
        "with CE (no detail specified)",
        "channel V intrinsic (z=0) CE not specified",
    }
    detail_cols = [
        ch["name"] for ch in CE_DETAILED_CHANNELS
        if ch["name"] not in broad_cols and ch["name"] in df.columns
    ]
    has_detail = df[detail_cols].sum(axis=1) > 0
    return df.loc[has_detail].reset_index(drop=True)


def build_hover_text(model_name, dco_label, table_df):
    """Build an HTML hover string for one model, pulling data from the table CSV."""
    prefix = DCO_PREFIX[dco_label]

    # Match on stripped model name to handle trailing-space mismatches
    mask = table_df["Model"].astype(str).str.strip() == str(model_name).strip()
    matching = table_df[mask]

    if matching.empty:
        return f"<b>{model_name}</b><br><i>No table entry found</i>"

    row = matching.iloc[0]
    paper = row.get("Paper", "N/A")

    lines = [
        f"<b>Paper:</b> {paper}",
        f"<b>Model:</b> {model_name}",
        "<b>──────────────────────────</b>",
    ]

    for col_tmpl, label in HOVER_COLS:
        col = col_tmpl.format(p=prefix)
        if col in row.index:
            val = row[col]
            if pd.isna(val):
                lines.append(f"{label}: N/A")
            elif label.startswith("Rate"):
                lines.append(f"<b>{label}:</b> {val:.2f}")
            else:
                lines.append(f"{label}: {val:.4f}")
        else:
            lines.append(f"{label}: N/A")

    return "<br>".join(lines)

In [ ]:
# ── Author colour map (shared across all calls) ──────────────────────────────

def build_author_color_map(specs_df):
    """Assign a distinct colour to each unique label_author."""
    # Qualitative palette with 20+ distinct colours
    palette_20 = [
        "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
        "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf",
        "#aec7e8", "#ffbb78", "#98df8a", "#ff9896", "#c5b0d5",
        "#c49c94", "#f7b6d2", "#c7c7c7", "#dbdb8d", "#9edae5",
        "#393b79", "#637939", "#8c6d31", "#843c39", "#7b4173",
    ]
    authors = sorted(specs_df["label_author"].dropna().astype(str).str.strip().unique())
    return {a: palette_20[i % len(palette_20)] for i, a in enumerate(authors)}


def make_interactive_figure(
    dco_label="BH-BH",
    fc_key="CE_simple",
    sort_column="fraction without common envelope",
    save_path=None,
    show=True,
):
    """
    Build an interactive Plotly figure matching the 3-panel matplotlib layout:
      Col 1 – simulation properties (author, code, tracks, parameters)
      Col 2 – stacked horizontal bar chart of formation channel fractions
      Col 3 – log-scale scatter of total merger rates

    Hover over any element to see Paper + all formation channel fractions.
    """
    # ── Load data ────────────────────────────────────────────────────────────
    rates_df, specs_df, table_df = load_all_data(dco_label)

    # Apply detailed-subset filter
    if fc_key == "CE_detailed":
        rates_df = filter_to_detailed_subset(rates_df)

    # Sort models (ascending = models with least CE at bottom, most CE at top)
    if sort_column in rates_df.columns:
        rates_df = rates_df.sort_values(sort_column, ascending=True).reset_index(drop=True)

    models   = rates_df["model"].tolist()
    n_models = len(models)
    y_vals   = list(range(n_models))  # 0 = bottom, N-1 = top

    # Align specs with sorted model order
    specs_idx = specs_df.set_index("model")
    specs_aligned = specs_idx.reindex(models).reset_index()

    # Build author colour map
    author_colors = build_author_color_map(specs_df)

    # Identify valid channels for this DCO's rates file
    channels = CE_SIMPLE_CHANNELS if fc_key == "CE_simple" else CE_DETAILED_CHANNELS
    valid_channels = [ch for ch in channels if ch["name"] in rates_df.columns]

    # Build per-model hover texts
    hover_texts = [build_hover_text(m, dco_label, table_df) for m in models]

    # ── Figure layout ────────────────────────────────────────────────────────
    fig = make_subplots(
        rows=1, cols=3,
        column_widths=[0.28, 0.57, 0.15],
        shared_yaxes=True,
        horizontal_spacing=0.01,
    )

    # ── Panel 1: Simulation properties ───────────────────────────────────────
    # Show coloured text per column: author | code | tracks | label_1 [| label_2]
    spec_cols_cfg = [
        {"col": "label_author", "x": 0.00, "use_author_color": True},
        {"col": "code",         "x": 0.22, "use_author_color": False},
        {"col": "stellar_tracks","x": 0.44, "use_author_color": False},
        {"col": "label_1",      "x": 0.66, "use_author_color": False},
        {"col": "label_2",      "x": 0.88, "use_author_color": False},
    ]

    for cfg in spec_cols_cfg:
        col_name = cfg["col"]
        if col_name not in specs_aligned.columns:
            continue

        x_pos   = cfg["x"]
        texts   = []
        colors  = []

        for ind_m, model in enumerate(models):
            raw = specs_aligned[col_name].iloc[ind_m]
            val = str(raw).strip() if pd.notna(raw) and str(raw).strip() not in ("", "nan") else ""
            texts.append(val)

            if cfg["use_author_color"]:
                author_raw = specs_aligned["label_author"].iloc[ind_m]
                author     = str(author_raw).strip() if pd.notna(author_raw) else ""
                colors.append(author_colors.get(author, "#555555"))
            else:
                colors.append("#333333")

        fig.add_trace(
            go.Scatter(
                x=[x_pos] * n_models,
                y=y_vals,
                mode="text",
                text=texts,
                textposition="middle right",
                textfont=dict(size=9, color=colors),
                customdata=hover_texts,
                hovertemplate="%{customdata}<extra></extra>",
                showlegend=False,
                name=col_name,
            ),
            row=1, col=1,
        )

    # Column headers in panel 1 (drawn as annotations relative to the x2 axis)
    col_headers = [
        (0.00, "author"),
        (0.22, "code"),
        (0.44, "tracks"),
        (0.66, "parameters"),
        (0.88, ""),
    ]

    # ── Panel 2: Formation channel fraction stacked bars ──────────────────────
    for ch in valid_channels:
        fracs = rates_df[ch["name"]].values.tolist()
        fig.add_trace(
            go.Bar(
                y=y_vals,
                x=fracs,
                name=ch["label"],
                orientation="h",
                marker_color=ch["color"],
                marker_line=dict(width=0.3, color="rgba(0,0,0,0.3)"),
                width=0.75,
                customdata=hover_texts,
                hovertemplate="%{customdata}<extra></extra>",
            ),
            row=1, col=2,
        )

    # ── Panel 3: Total merger rates ───────────────────────────────────────────
    total_rates = rates_df[TOTAL_RATE_COL].values.tolist()

    fig.add_trace(
        go.Scatter(
            y=y_vals,
            x=total_rates,
            mode="markers",
            marker=dict(symbol="diamond", size=8, color="black"),
            name="Total rate",
            customdata=hover_texts,
            hovertemplate="%{customdata}<extra></extra>",
        ),
        row=1, col=3,
    )

    # ── Dotted horizontal grid lines (one per model, in bar panel) ────────────
    for y_v in y_vals:
        fig.add_hline(
            y=y_v,
            line=dict(color="lightgray", width=0.5, dash="dot"),
            row=1, col=2,
        )

    # ── Observed GW merger rate shaded band ───────────────────────────────────
    gw_min, gw_max = GW_RATE_BANDS[dco_label]
    fig.add_vrect(
        x0=gw_min, x1=gw_max,
        fillcolor="lightgray", opacity=0.8,
        layer="below", line_width=0,
        row=1, col=3,
    )

    # ── Axes configuration ────────────────────────────────────────────────────
    dco_title = {"BH-BH": "BBH", "BH-NS": "BHNS", "NS-NS": "BNS"}[dco_label]
    fig_title = f"{dco_title} — Formation Channels ({fc_key.replace('_', ' ')})"

    # Y range and tick labels (model names on the left y-axis)
    y_range = [-1, n_models + 0.25]

    fig.update_yaxes(
        range=y_range,
        tickvals=y_vals,
        ticktext=models,
        tickfont=dict(size=8),
        row=1, col=1,
    )
    fig.update_yaxes(range=y_range, row=1, col=2)
    fig.update_yaxes(range=y_range, row=1, col=3)

    # Panel 1 x-axis (hidden)
    fig.update_xaxes(
        range=[0, 1],
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        showline=False,
        row=1, col=1,
    )

    # Panel 2 x-axis (fraction 0–1)
    fig.update_xaxes(
        range=[0, 1],
        title_text="fraction",
        title_font=dict(size=13),
        tickvals=[0, 0.2, 0.4, 0.6, 0.8, 1.0],
        showgrid=True,
        gridcolor="lightgray",
        row=1, col=2,
    )

    # Panel 3 x-axis (log-scale rate)
    x_lo, x_hi = XLIMS_RATE[dco_label]
    fig.update_xaxes(
        type="log",
        range=[np.log10(x_lo), np.log10(x_hi)],
        title_text="R [Gpc⁻³ yr⁻¹]",
        title_font=dict(size=13),
        showgrid=True,
        gridcolor="lightgray",
        row=1, col=3,
    )

    # ── Column header annotations for panel 1 ────────────────────────────────
    # We add these as figure-level annotations anchored to xaxis1
    header_y = n_models + 0.1  # just above the top model
    for x_pos, header_text in col_headers:
        fig.add_annotation(
            x=x_pos, y=header_y,
            text=f"<b>{header_text}</b>",
            showarrow=False,
            xref="x1", yref="y1",
            xanchor="left", yanchor="bottom",
            font=dict(size=10),
        )

    # DCO label annotation in centre of panel 2
    fig.add_annotation(
        x=0.5, y=1.01,
        text=f"<b>{dco_title}</b>",
        showarrow=False,
        xref="x2 domain", yref="paper",
        xanchor="center", yanchor="bottom",
        font=dict(size=20),
    )

    # ── Global layout ─────────────────────────────────────────────────────────
    fig.update_layout(
        title=dict(text=fig_title, x=0.5, font=dict(size=16)),
        barmode="stack",
        height=max(600, 22 * n_models + 80),
        width=1500,
        plot_bgcolor="white",
        paper_bgcolor="white",
        hoverlabel=dict(bgcolor="white", font_size=11, namelength=-1),
        legend=dict(
            x=1.01, y=1,
            orientation="v",
            font=dict(size=10),
            tracegroupgap=2,
        ),
        margin=dict(l=10, r=200, t=60, b=40),
    )

    if save_path:
        fig.write_html(save_path)
        print(f"Saved: {save_path}")

    if show:
        fig.show()

    return fig

## BH-BH (Fig 3)

In [ ]:
# Fig3 — BH-BH simple (CE vs no-CE)
fig_bhbh_simple = make_interactive_figure(
    dco_label="BH-BH",
    fc_key="CE_simple",
    sort_column="fraction without common envelope",
    save_path=path_save + "Fig3_BH-BH_simple_interactive.html",
)

In [ ]:
# Fig3 — BH-BH detailed formation channels
fig_bhbh_detailed = make_interactive_figure(
    dco_label="BH-BH",
    fc_key="CE_detailed",
    sort_column="fraction without common envelope",
    save_path=path_save + "Fig3_BH-BH_detailed_interactive.html",
)

## BH-NS (Fig 4)

In [ ]:
# Fig4 — BH-NS simple
fig_bhns_simple = make_interactive_figure(
    dco_label="BH-NS",
    fc_key="CE_simple",
    sort_column="fraction without common envelope",
    save_path=path_save + "Fig4_BH-NS_simple_interactive.html",
)

In [ ]:
# Fig4 — BH-NS detailed
fig_bhns_detailed = make_interactive_figure(
    dco_label="BH-NS",
    fc_key="CE_detailed",
    sort_column="fraction without common envelope",
    save_path=path_save + "Fig4_BH-NS_detailed_interactive.html",
)

## NS-NS (Fig 5)

In [ ]:
# Fig5 — NS-NS simple
fig_nsns_simple = make_interactive_figure(
    dco_label="NS-NS",
    fc_key="CE_simple",
    sort_column="fraction without common envelope",
    save_path=path_save + "Fig5_NS-NS_simple_interactive.html",
)

In [ ]:
# Fig5 — NS-NS detailed
fig_nsns_detailed = make_interactive_figure(
    dco_label="NS-NS",
    fc_key="CE_detailed",
    sort_column="fraction without common envelope",
    save_path=path_save + "Fig5_NS-NS_detailed_interactive.html",
)